# Spatio-temporal trajectory analysis

Builds object trajectories from the real-world coordinates produced by
`03_coordinate_mapping/process_yolo_output.py` and checks which road users were
close to the shuttle around the time of an event (e.g. a hard brake detected
from the vehicle's sensor data).

## Parameters

In [ ]:
CSV_PATH = '../03_coordinate_mapping/coords_3d.csv'   # output of step 3
EVENT_TIME = '2022-10-27 17:10:51'   # timestamp of the event to investigate (e.g. hard brake)
WINDOW_S = 2                          # seconds to inspect before/after the event
DISTANCE_THRESHOLD = 15               # metres: only report objects closer than this
X_TOLERANCE = 2                       # metres: |dx| below this counts as "next to"
PARKED_MAX_LENGTH = 12                # metres: car trajectories shorter than this are considered parked
EGO_CLASS = 'shuttle'                 # class name of the vehicle of interest

In [ ]:
import pandas as pd
import movingpandas as mpd
import matplotlib.pyplot as plt
import hvplot.pandas  # noqa: F401  (enables interactive .hvplot())
from holoviews import opts
from shapely.geometry import Point
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

## Load data and build trajectories

In [ ]:
df = pd.read_csv(CSV_PATH, parse_dates=['ts'])
df.head()

In [ ]:
# x_3d / y_3d are local metric coordinates; EPSG:3857 is only used so lengths/speeds are reported in metres
tc = mpd.TrajectoryCollection(df, ['classname', 'obj_id'], t='ts', x='x_3d', y='y_3d', crs='EPSG:3857')

def by_class(name):
    return [traj for traj in tc.trajectories if traj.id[0] == name]

cars = by_class('car')
parked_cars_tc = mpd.TrajectoryCollection([t for t in cars if t.get_length() < PARKED_MAX_LENGTH])
moving_cars_tc = mpd.TrajectoryCollection([t for t in cars if t.get_length() >= PARKED_MAX_LENGTH])
people_tc = mpd.TrajectoryCollection(by_class('person'))
shuttle_tc = mpd.TrajectoryCollection(by_class(EGO_CLASS))

print(f"parked cars: {len(parked_cars_tc)} | moving cars: {len(moving_cars_tc)} | "
      f"people: {len(people_tc)} | {EGO_CLASS}: {len(shuttle_tc)}")

## Plot trajectories per class

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (title, collection, label) in zip(axes, [
        ('People trajectories', people_tc, 'person'),
        ('Moving car trajectories', moving_cars_tc, 'car'),
        (f'{EGO_CLASS.capitalize()} trajectories', shuttle_tc, EGO_CLASS)]):
    cmap = plt.get_cmap('tab10', max(len(collection), 1))
    for i, traj in enumerate(collection.trajectories):
        traj.plot(ax=ax, label=f'{label} {traj.id[1]}', color=cmap(i))
    ax.set_title(title)
    ax.set_xlabel('x [m]')
    ax.set_ylabel('y [m]')
    if len(collection):
        ax.legend()

plt.show()

## Relative position of moving cars around the event

For every timestamp in the window around `EVENT_TIME`, report the cars within
`DISTANCE_THRESHOLD` metres of the shuttle and whether they are in front of,
behind or next to it (along the x axis of the world frame).

In [ ]:
shuttle_traj = shuttle_tc.trajectories[0]
T = datetime.strptime(EVENT_TIME, "%Y-%m-%d %H:%M:%S")
timestamps_to_compare = [T + timedelta(seconds=i) for i in range(-WINDOW_S, WINDOW_S + 1)]

nearby = []
for compare_T in timestamps_to_compare:
    if not (shuttle_traj.get_start_time() <= compare_T <= shuttle_traj.get_end_time()):
        continue
    shuttle_point = Point(shuttle_traj.get_position_at(compare_T))

    for car_traj in moving_cars_tc.trajectories:
        if not (car_traj.get_start_time() <= compare_T <= car_traj.get_end_time()):
            continue
        car_point = Point(car_traj.get_position_at(compare_T))
        distance = shuttle_point.distance(car_point)

        if distance < DISTANCE_THRESHOLD:
            if abs(shuttle_point.x - car_point.x) <= X_TOLERANCE:
                position = 'next to'
            elif shuttle_point.x > car_point.x:
                position = 'in front of'
            else:
                position = 'behind'
            nearby.append({'ts': compare_T, 'object': car_traj.id, 'position': position,
                           'distance_m': round(distance, 2)})

pd.DataFrame(nearby)

## Speed along the trajectories

In [ ]:
opts.defaults(opts.Overlay(active_tools=['wheel_zoom'], frame_width=500, frame_height=400))
tc.add_speed(overwrite=True)

ids_of_interest = [shuttle_traj.id[1]] + sorted({obj[1] for obj in pd.DataFrame(nearby).get('object', [])})
tc.filter('obj_id', ids_of_interest).hvplot(c='speed', line_width=5, colorbar=True, clim=(0, 10),
                                            tiles=None, geo=False, frame_width=700, frame_height=350)